In [1]:
# ==========================================================
# BLOQUE 1. Imports y configuración general
# Admisión alumnado internacional
# ==========================================================

import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup

from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==========================================================
# BLOQUE 2. Constantes
# ==========================================================

URL = (
    "https://www.upv.es/admision/internacional/"
)

HEADERS = {
    "User-Agent":
        "Mozilla/5.0 (compatible; UPV-Admision-Bot/1.0)"
}

PAUSA = 0.3

# Dominio/base de la microweb que queremos extraer
DOMINIO_BASE = (
    "https://www.upv.es/admision/internacional/"
)

# Rutas que no forman parte del contenido que queremos extraer
RUTAS_EXCLUIDAS = [
    "/wp-login.php",
    "/admision/internacional/va/",
    "/admision/internacional/en/"
]

In [3]:
# ==========================================================
# BLOQUE 3. Ruta del JSON
# ==========================================================

NOMBRE_PROGRAMA = (
    "Extrae_Admision_Internacional.ipynb"
)

ruta_programa = None

for root, dirs, files in os.walk(
    "/content/drive/MyDrive"
):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root

        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(
    ruta_programa,
    "JSONs"
)


os.makedirs(
    CARPETA_JSON,
    exist_ok=True
)


RUTA_JSON = os.path.join(
    CARPETA_JSON,
    "admision_internacional.json"
)


print("Directorio del proyecto:")
print(ruta_programa)

print()

print("JSON:")
print(RUTA_JSON)

Directorio del proyecto:
/content/drive/MyDrive/TFG Teleco

JSON:
/content/drive/MyDrive/TFG Teleco/JSONs/admision_internacional.json


In [30]:
# ==========================================================
# BLOQUE 4. Funciones auxiliares
# ==========================================================

def limpiar_texto(elemento):

    if elemento is None:
        return ""

    return " ".join(
        elemento.stripped_strings
    )


def url_absoluta(url):

    if not url:
        return ""

    return urljoin(
        URL,
        url
    )


def normalizar_url(url):

    if not url:
        return ""

    url = url_absoluta(url)

    partes = urlparse(url)

    return urlunparse((
        partes.scheme,
        partes.netloc,
        partes.path.rstrip("/"),
        "",
        partes.query,
        ""
    ))


def obtener_descripcion(contenedor):

    if contenedor is None:
        return ""

    parrafos = contenedor.find_all(
        "p",
        recursive=False
    )

    texto = " ".join(
        texto_limpio(p)
        for p in parrafos
    )

    return texto.strip()

In [24]:
# ============================================================
# B5 - CONTENEDORES DE BLOQUES / CALENDARIOS
# ============================================================

def extraer_bloques_contenedor(contenedor):
    """
    Extrae contenedores del tipo upv-blocks y organiza su contenido
    en subsecciones semánticas.

    Caso específico:
        - Calendario para grados
        - Calendario para másteres
        - Calendario para doctorado
    """

    subsecciones = []

    # --------------------------------------------------------
    # Buscar bloques internos
    # --------------------------------------------------------
    bloques_dom = contenedor.find_all(
        ["div", "section", "article"],
        recursive=True
    )

    bloques_detectados = []

    for bloque_dom in bloques_dom:

        clases = bloque_dom.get("class", [])

        # Solo elementos que realmente parezcan bloques
        if not any(
            c in clases
            for c in [
                "upv-block",
                "upv-box",
                "wp-block-group"
            ]
        ):
            continue

        titulo = extraer_titulo(bloque_dom)

        texto = limpiar_texto(
            bloque_dom.get_text(" ", strip=True)
        )

        enlaces = extraer_enlaces(bloque_dom)

        if not titulo and not texto and not enlaces:
            continue

        bloques_detectados.append({
            "titulo": titulo,
            "texto": texto,
            "enlaces": enlaces
        })

    # --------------------------------------------------------
    # Si no se han encontrado bloques explícitos, no crear
    # estructura artificial.
    # --------------------------------------------------------
    if not bloques_detectados:
        return []

    # --------------------------------------------------------
    # Identificar bloques por su contenido/título
    # --------------------------------------------------------
    grados = []
    masteres = []
    doctorado = []

    for bloque in bloques_detectados:

        titulo = normalizar_espacios(
            bloque["titulo"]
        ).lower()

        texto = normalizar_espacios(
            bloque["texto"]
        )

        texto_lower = texto.lower()

        # ----------------------------------------------------
        # Grados
        # ----------------------------------------------------
        if (
            "grado" in titulo
            or "grado" in texto_lower
            or "alumnado internacional" in texto_lower
            and "finales de julio" in texto_lower
        ):
            grados.append(bloque)
            continue

        # ----------------------------------------------------
        # Másteres
        # ----------------------------------------------------
        if (
            "máster" in titulo
            or "master" in titulo
            or "máster" in texto_lower
            or "master" in texto_lower
        ):
            masteres.append(bloque)
            continue

        # ----------------------------------------------------
        # Doctorado
        # ----------------------------------------------------
        if (
            "doctorado" in titulo
            or "doctorado" in texto_lower
        ):
            doctorado.append(bloque)
            continue

    # --------------------------------------------------------
    # Si la detección por contenido no funciona, utilizar
    # el orden natural de los bloques.
    # --------------------------------------------------------
    if not grados and not masteres and not doctorado:

        grupos = []

        indice = 0

        while indice < len(bloques_detectados):

            if indice == 0:
                grupo = bloques_detectados[indice:indice + 2]
                grupos.append(
                    ("Calendario para grados", grupo)
                )

            elif indice == 2:
                grupo = bloques_detectados[indice:indice + 2]
                grupos.append(
                    ("Calendario para másteres", grupo)
                )

            elif indice == 4:
                grupo = bloques_detectados[indice:indice + 3]
                grupos.append(
                    ("Calendario para doctorado", grupo)
                )

            indice += len(grupo)

        for titulo_grupo, bloques in grupos:

            subsecciones.append({
                "titulo": titulo_grupo,
                "texto": "",
                "enlaces": [],
                "bloques": [
                    limpiar_bloque(b)
                    for b in bloques
                ]
            })

        return subsecciones

    # --------------------------------------------------------
    # Crear subsecciones semánticas
    # --------------------------------------------------------

    if grados:

        subsecciones.append({
            "titulo": "Calendario para grados",
            "texto": "",
            "enlaces": obtener_enlaces_subseccion(grados),
            "bloques": [
                limpiar_bloque(b)
                for b in grados
            ]
        })

    if masteres:

        subsecciones.append({
            "titulo": "Calendario para másteres",
            "texto": "",
            "enlaces": obtener_enlaces_subseccion(masteres),
            "bloques": [
                limpiar_bloque(b)
                for b in masteres
            ]
        })

    if doctorado:

        subsecciones.append({
            "titulo": "Calendario para doctorado",
            "texto": "",
            "enlaces": obtener_enlaces_subseccion(doctorado),
            "bloques": [
                limpiar_bloque(b)
                for b in doctorado
            ]
        })

    return subsecciones


# ============================================================
# LIMPIEZA DE BLOQUES
# ============================================================

def limpiar_bloque(bloque):

    titulo = normalizar_espacios(
        bloque.get("titulo", "")
    )

    texto = normalizar_espacios(
        bloque.get("texto", "")
    )

    # --------------------------------------------------------
    # Evitar duplicación del título al principio del texto
    # --------------------------------------------------------

    if titulo:
        prefijo = titulo.strip()

        if texto.lower().startswith(
            prefijo.lower()
        ):
            texto = texto[len(prefijo):].strip()

    # --------------------------------------------------------
    # Limpiar títulos residuales
    # --------------------------------------------------------

    for prefijo in [
        "Preinscripción",
        "Matrícula",
        "Resolución"
    ]:

        if texto.lower().startswith(
            prefijo.lower()
        ):
            texto = texto[len(prefijo):].strip()

    return {
        "titulo": titulo,
        "texto": texto,
        "enlaces": bloque.get("enlaces", [])
    }


# ============================================================
# ENLACES DE UNA SUBSECCIÓN
# ============================================================

def obtener_enlaces_subseccion(bloques):

    enlaces = []

    for bloque in bloques:

        for enlace in bloque.get("enlaces", []):

            if enlace not in enlaces:
                enlaces.append(enlace)

    return enlaces

In [25]:
# ==========================================================
# BLOQUE 6. Validación y normalización del JSON
# Admisión Internacional
# ==========================================================

import json
from urllib.parse import urlparse


# ==========================================================
# 1. VALIDACIÓN DE LA ESTRUCTURA
# ==========================================================

print("=" * 70)
print("VALIDACIÓN DE LA ESTRUCTURA")
print("=" * 70)


# ----------------------------------------------------------
# Estructura principal
# ----------------------------------------------------------

if not isinstance(datos, dict):
    raise TypeError(
        "El objeto 'datos' no es un diccionario."
    )


if "padres" not in datos:
    raise ValueError(
        "El JSON no contiene la clave 'padres'."
    )


if not isinstance(datos["padres"], list):
    raise TypeError(
        "La clave 'padres' debe contener una lista."
    )


if len(datos["padres"]) == 0:
    raise ValueError(
        "No se ha encontrado ningún padre."
    )


print("✓ Estructura principal: OK")


# ==========================================================
# 2. VALIDACIÓN DEL PADRE
# ==========================================================

padre = datos["padres"][0]


campos_padre = [
    "titulo",
    "url",
    "secciones"
]


for campo in campos_padre:

    if campo not in padre:
        raise ValueError(
            f"Falta el campo '{campo}' en el padre."
        )


print("✓ Campos del padre: OK")


# ----------------------------------------------------------
# Validación de URL
# ----------------------------------------------------------

url = padre["url"]

parsed = urlparse(url)


if not parsed.scheme or not parsed.netloc:

    raise ValueError(
        f"La URL no parece válida: {url}"
    )


print(
    "✓ URL:",
    url
)


# ==========================================================
# 3. VALIDACIÓN DE LAS SECCIONES
# ==========================================================

secciones = padre["secciones"]


if not isinstance(secciones, list):

    raise TypeError(
        "El campo 'secciones' debe ser una lista."
    )


print(
    "✓ Secciones encontradas:",
    len(secciones)
)


# ==========================================================
# 4. VALIDACIÓN RECURSIVA
# ==========================================================

def validar_bloque(bloque, ruta="bloque"):

    if not isinstance(bloque, dict):

        raise TypeError(
            f"{ruta} no es un diccionario."
        )


    campos = [
        "clases",
        "titulo",
        "texto",
        "enlaces"
    ]


    for campo in campos:

        if campo not in bloque:

            raise ValueError(
                f"Falta '{campo}' en {ruta}."
            )


    if not isinstance(
        bloque["clases"],
        list
    ):

        raise TypeError(
            f"{ruta}.clases debe ser una lista."
        )


    if not isinstance(
        bloque["enlaces"],
        list
    ):

        raise TypeError(
            f"{ruta}.enlaces debe ser una lista."
        )


    for i, enlace in enumerate(
        bloque["enlaces"]
    ):

        if "texto" not in enlace:
            raise ValueError(
                f"Falta texto en {ruta}.enlaces[{i}]"
            )

        if "url" not in enlace:
            raise ValueError(
                f"Falta url en {ruta}.enlaces[{i}]"
            )


    if "bloques" in bloque:

        if not isinstance(
            bloque["bloques"],
            list
        ):

            raise TypeError(
                f"{ruta}.bloques debe ser una lista."
            )


        for i, sub_bloque in enumerate(
            bloque["bloques"]
        ):

            validar_bloque(
                sub_bloque,
                f"{ruta}.bloques[{i}]"
            )


    if "subsecciones" in bloque:

        if not isinstance(
            bloque["subsecciones"],
            list
        ):

            raise TypeError(
                f"{ruta}.subsecciones debe ser una lista."
            )


        for i, subseccion in enumerate(
            bloque["subsecciones"]
        ):

            validar_bloque(
                subseccion,
                f"{ruta}.subsecciones[{i}]"
            )


for i, seccion in enumerate(
    secciones
):

    validar_bloque(
        seccion,
        f"secciones[{i}]"
    )


print(
    "✓ Estructura interna: OK"
)


# ==========================================================
# 5. ESTADÍSTICAS DE LA ESTRUCTURA
# ==========================================================

estadisticas = {

    "secciones": 0,

    "subsecciones": 0,

    "bloques": 0,

    "enlaces": 0

}


def contar_elementos(elemento):

    if not isinstance(
        elemento,
        dict
    ):
        return


    estadisticas["secciones"] += (
        1
        if elemento.get("tipo") in [
            "contenido",
            "contenedor",
            "seccion"
        ]
        else 0
    )


    estadisticas["subsecciones"] += len(
        elemento.get(
            "subsecciones",
            []
        )
    )


    estadisticas["bloques"] += len(
        elemento.get(
            "bloques",
            []
        )
    )


    estadisticas["enlaces"] += len(
        elemento.get(
            "enlaces",
            []
        )
    )


    for subseccion in elemento.get(
        "subsecciones",
        []
    ):

        contar_elementos(
            subseccion
        )


    for bloque in elemento.get(
        "bloques",
        []
    ):

        contar_elementos(
            bloque
        )


for seccion in secciones:

    contar_elementos(
        seccion
    )


print()
print("ESTADÍSTICAS")
print(
    "  Secciones:",
    len(secciones)
)
print(
    "  Subsecciones:",
    estadisticas["subsecciones"]
)
print(
    "  Bloques:",
    estadisticas["bloques"]
)
print(
    "  Enlaces:",
    estadisticas["enlaces"]
)


# ==========================================================
# 6. NORMALIZACIÓN
# ==========================================================

print()
print("=" * 70)
print("NORMALIZACIÓN")
print("=" * 70)


def normalizar_texto(texto):

    if not texto:
        return ""

    return " ".join(
        texto.split()
    )


def normalizar_url(url):

    if not url:
        return ""

    url = url.strip()

    # Si accidentalmente se ha introducido
    # formato Markdown:
    #
    # [https://ejemplo.com](https://ejemplo.com)
    #
    # nos quedamos únicamente con la URL.

    if url.startswith("[") and "](" in url:

        inicio = url.find("](") + 2
        fin = url.rfind(")")

        if fin > inicio:

            url = url[
                inicio:fin
            ]


    return url


def normalizar_enlace(enlace):

    return {

        "texto": normalizar_texto(
            enlace.get(
                "texto",
                ""
            )
        ),

        "url": normalizar_url(
            enlace.get(
                "url",
                ""
            )
        )

    }


# ==========================================================
# 7. NORMALIZACIÓN DE BLOQUES
# ==========================================================

def normalizar_bloque(bloque):

    resultado = {

        "titulo": normalizar_texto(
            bloque.get(
                "titulo",
                ""
            )
        ),

        "texto": normalizar_texto(
            bloque.get(
                "texto",
                ""
            )
        ),

        "enlaces": [
            normalizar_enlace(
                enlace
            )
            for enlace in bloque.get(
                "enlaces",
                []
            )
        ]

    }


    # ------------------------------------------------------
    # Bloques internos
    # ------------------------------------------------------

    bloques = bloque.get(
        "bloques",
        []
    )


    if bloques:

        resultado["bloques"] = [

            normalizar_bloque(
                sub_bloque
            )

            for sub_bloque in bloques

        ]


    # ------------------------------------------------------
    # Subsecciones
    # ------------------------------------------------------

    subsecciones = bloque.get(
        "subsecciones",
        []
    )


    if subsecciones:

        resultado["subsecciones"] = [

            normalizar_bloque(
                subseccion
            )

            for subseccion in subsecciones

        ]


    return resultado


# ==========================================================
# 8. NORMALIZACIÓN DE SECCIONES
# ==========================================================

secciones_normalizadas = []


for seccion in secciones:

    clases = seccion.get(
        "clases",
        []
    )


    # ------------------------------------------------------
    # Determinar tipo
    # ------------------------------------------------------

    if "upv-blocks" in clases:

        tipo = "contenedor"

    elif "upv-boxes" in clases:

        tipo = "seccion"

    else:

        tipo = "contenido"


    normalizada = {

        "tipo": tipo,

        "clases": clases,

        "titulo": normalizar_texto(
            seccion.get(
                "titulo",
                ""
            )
        ),

        "texto": normalizar_texto(
            seccion.get(
                "texto",
                ""
            )
        ),

        "enlaces": [
            normalizar_enlace(
                enlace
            )
            for enlace in seccion.get(
                "enlaces",
                []
            )
        ]

    }


    # ------------------------------------------------------
    # Subsecciones
    # ------------------------------------------------------

    subsecciones = seccion.get(
        "subsecciones",
        []
    )


    if subsecciones:

        normalizada["subsecciones"] = []


        for subseccion in subsecciones:

            sub = {

                "titulo": normalizar_texto(
                    subseccion.get(
                        "titulo",
                        ""
                    )
                ),

                "texto": "",

                "enlaces": [
                    normalizar_enlace(
                        enlace
                    )
                    for enlace in subseccion.get(
                        "enlaces",
                        []
                    )
                ]

            }


            # --------------------------------------------------
            # IMPORTANTE:
            #
            # Para subsecciones que contienen bloques,
            # no repetimos el texto general de la subsección.
            # El contenido queda representado por sus bloques.
            # --------------------------------------------------

            bloques = subseccion.get(
                "bloques",
                []
            )


            if bloques:

                sub["bloques"] = [

                    normalizar_bloque(
                        bloque
                    )

                    for bloque in bloques

                ]

            else:

                sub["texto"] = normalizar_texto(
                    subseccion.get(
                        "texto",
                        ""
                    )
                )


            normalizada[
                "subsecciones"
            ].append(
                sub
            )


    # ------------------------------------------------------
    # Bloques directos
    # ------------------------------------------------------

    bloques = seccion.get(
        "bloques",
        []
    )


    if bloques:

        normalizada["bloques"] = [

            normalizar_bloque(
                bloque
            )

            for bloque in bloques

        ]


    secciones_normalizadas.append(
        normalizada
    )


# ==========================================================
# 9. JSON DEFINITIVO
# ==========================================================

datos_normalizados = {

    "padres": [

        {

            "titulo": normalizar_texto(
                padre["titulo"]
            ),

            "url": normalizar_url(
                padre["url"]
            ),

            "secciones":
                secciones_normalizadas

        }

    ]

}


# ==========================================================
# 10. COMPROBACIÓN FINAL DE URL
# ==========================================================

url_final = datos_normalizados[
    "padres"
][0]["url"]


print()
print(
    "URL final:",
    url_final
)


if url_final != URL:

    raise ValueError(
        "La URL del JSON no coincide con URL."
    )


print(
    "✓ URL correctamente normalizada."
)


# ==========================================================
# 11. MOSTRAR JSON DEFINITIVO
# ==========================================================

print()
print("=" * 70)
print("JSON DEFINITIVO NORMALIZADO")
print("=" * 70)


print(
    json.dumps(
        datos_normalizados,
        ensure_ascii=False,
        indent=4
    )
)


# ==========================================================
# 12. RESUMEN FINAL
# ==========================================================

print()
print("=" * 70)
print("RESUMEN FINAL")
print("=" * 70)

print(
    "Padre:",
    datos_normalizados[
        "padres"
    ][0]["titulo"]
)

print(
    "URL:",
    datos_normalizados[
        "padres"
    ][0]["url"]
)

print(
    "Secciones:",
    len(
        datos_normalizados[
            "padres"
        ][0]["secciones"]
    )
)

for i, seccion in enumerate(
    datos_normalizados[
        "padres"
    ][0]["secciones"],
    start=1
):

    print()
    print(
        f"SECCIÓN {i}"
    )

    print(
        "  Tipo:",
        seccion["tipo"]
    )

    print(
        "  Título:",
        seccion["titulo"]
    )

    print(
        "  Subsecciones:",
        len(
            seccion.get(
                "subsecciones",
                []
            )
        )
    )

    print(
        "  Bloques:",
        len(
            seccion.get(
                "bloques",
                []
            )
        )
    )

    for subseccion in seccion.get(
        "subsecciones",
        []
    ):

        print(
            "    -",
            subseccion["titulo"]
        )

        print(
            "      bloques:",
            len(
                subseccion.get(
                    "bloques",
                    []
                )
            )
        )

VALIDACIÓN DE LA ESTRUCTURA
✓ Estructura principal: OK
✓ Campos del padre: OK
✓ URL: https://www.upv.es/admision/internacional/
✓ Secciones encontradas: 8
✓ Estructura interna: OK

ESTADÍSTICAS
  Secciones: 8
  Subsecciones: 4
  Bloques: 8
  Enlaces: 13

NORMALIZACIÓN

URL final: https://www.upv.es/admision/internacional/
✓ URL correctamente normalizada.

JSON DEFINITIVO NORMALIZADO
{
    "padres": [
        {
            "titulo": "Alumnado internacional",
            "url": "https://www.upv.es/admision/internacional/",
            "secciones": [
                {
                    "tipo": "contenido",
                    "clases": [
                        "wp-block-group",
                        "alignfull",
                        "upv-text",
                        "upv-section",
                        "no-padding"
                    ],
                    "titulo": "",
                    "texto": "La Universitat Politècnica de València (UPV) es una universidad pública y de 

In [27]:
# ==========================================================
# BLOQUE 7. Generación Markdown base para RAG
# A partir del JSON final normalizado
# ==========================================================

import os
import re


# ==========================================================
# 1. RUTAS
# ==========================================================

directorio_base = os.path.join(
    ruta_programa,
    "ADMISION",
    "Internacional"
)


os.makedirs(
    directorio_base,
    exist_ok=True
)


# ==========================================================
# 2. PARÁMETROS
# ==========================================================

CATEGORIA = "admision"
NIVEL = "internacional"

INTRO_DOCUMENTO = (
    "Información sobre el proceso de admisión "
    "de alumnado internacional en la "
    "Universitat Politècnica de València."
)


# ==========================================================
# 3. LIMPIEZA DE NOMBRES DE ARCHIVO
# ==========================================================

def limpiar_nombre(nombre):

    if not nombre:
        return ""

    nombre = nombre.lower()


    cambios = {

        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
        "ñ": "n"

    }


    for viejo, nuevo in cambios.items():

        nombre = nombre.replace(
            viejo,
            nuevo
        )


    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )


    return nombre.strip("_")


# ==========================================================
# 4. GENERACIÓN DE NOMBRES ÚNICOS
# ==========================================================

def generar_nombre_seccion(
    titulo,
    indice,
    nombres_utilizados
):

    nombre = limpiar_nombre(
        titulo
    )


    if not nombre:

        nombre = f"seccion_{indice}"


    nombre_original = nombre

    contador = 2


    while nombre in nombres_utilizados:

        nombre = (
            f"{nombre_original}_{contador}"
        )

        contador += 1


    nombres_utilizados.add(
        nombre
    )


    return nombre


# ==========================================================
# 5. METADATOS YAML
# ==========================================================

def escribir_metadatos(
    f,
    tipo_documento,
    seccion=None
):

    f.write(
        "---\n"
    )

    f.write(
        "fuente: UPV\n"
    )

    f.write(
        f"categoria: {CATEGORIA}\n"
    )

    f.write(
        f"nivel: {NIVEL}\n"
    )

    f.write(
        f"tipo_documento: {tipo_documento}\n"
    )


    if seccion:

        f.write(
            f"seccion: {seccion}\n"
        )


    f.write(
        "---\n\n"
    )


# ==========================================================
# 6. ESCRIBIR ENLACES
# ==========================================================

def escribir_enlaces(
    f,
    enlaces,
    prefijo="-"
):

    for enlace in enlaces:

        texto = enlace.get(
            "texto",
            ""
        )

        url = enlace.get(
            "url",
            ""
        )


        if not url:

            continue


        if texto:

            f.write(
                f"{prefijo} "
                f"{texto}: "
                f"{url}\n"
            )

        else:

            f.write(
                f"{prefijo} "
                f"{url}\n"
            )


    if enlaces:

        f.write(
            "\n"
        )


# ==========================================================
# 7. ESCRIBIR UN BLOQUE
# ==========================================================

def escribir_bloque(
    f,
    bloque,
    nivel=3
):

    titulo = bloque.get(
        "titulo",
        ""
    )


    texto = bloque.get(
        "texto",
        ""
    )


    enlaces = bloque.get(
        "enlaces",
        []
    )


    # ------------------------------------------------------
    # Título
    # ------------------------------------------------------

    if titulo:

        encabezado = "#" * nivel

        f.write(
            f"{encabezado} "
            f"{titulo}\n\n"
        )


    # ------------------------------------------------------
    # Texto
    # ------------------------------------------------------

    if texto:

        f.write(
            texto
            +
            "\n\n"
        )


    # ------------------------------------------------------
    # Enlaces
    # ------------------------------------------------------

    escribir_enlaces(
        f,
        enlaces
    )


    # ------------------------------------------------------
    # Bloques internos
    # ------------------------------------------------------

    for sub_bloque in bloque.get(
        "bloques",
        []
    ):

        escribir_bloque(
            f,
            sub_bloque,
            nivel=min(
                nivel + 1,
                6
            )
        )


    # ------------------------------------------------------
    # Subsecciones internas
    # ------------------------------------------------------

    for subseccion in bloque.get(
        "subsecciones",
        []
    ):

        escribir_subseccion(
            f,
            subseccion,
            nivel=min(
                nivel + 1,
                6
            )
        )


# ==========================================================
# 8. ESCRIBIR UNA SUBSECCIÓN
# ==========================================================

def escribir_subseccion(
    f,
    subseccion,
    nivel=2
):

    titulo = subseccion.get(
        "titulo",
        ""
    )


    texto = subseccion.get(
        "texto",
        ""
    )


    enlaces = subseccion.get(
        "enlaces",
        []
    )


    # ------------------------------------------------------
    # Título
    # ------------------------------------------------------

    if titulo:

        encabezado = "#" * nivel

        f.write(
            f"{encabezado} "
            f"{titulo}\n\n"
        )


    # ------------------------------------------------------
    # Texto
    # ------------------------------------------------------

    if texto:

        f.write(
            texto
            +
            "\n\n"
        )


    # ------------------------------------------------------
    # Enlaces
    # ------------------------------------------------------

    escribir_enlaces(
        f,
        enlaces
    )


    # ------------------------------------------------------
    # Bloques
    # ------------------------------------------------------

    for bloque in subseccion.get(
        "bloques",
        []
    ):

        escribir_bloque(
            f,
            bloque,
            nivel=min(
                nivel + 1,
                6
            )
        )


    # ------------------------------------------------------
    # Subsecciones internas
    # ------------------------------------------------------

    for sub_subseccion in subseccion.get(
        "subsecciones",
        []
    ):

        escribir_subseccion(
            f,
            sub_subseccion,
            nivel=min(
                nivel + 1,
                6
            )
        )


# ==========================================================
# 9. ESCRIBIR UNA SECCIÓN
# ==========================================================

def escribir_seccion(
    f,
    seccion,
    padre
):

    titulo = seccion.get(
        "titulo",
        ""
    )


    texto = seccion.get(
        "texto",
        ""
    )


    enlaces = seccion.get(
        "enlaces",
        []
    )


    # ------------------------------------------------------
    # Título de sección
    # ------------------------------------------------------

    if titulo:

        f.write(
            f"# {titulo}\n\n"
        )


    else:

        f.write(
            "# Información de admisión\n\n"
        )


    # ------------------------------------------------------
    # Contexto del padre
    # ------------------------------------------------------

    if padre.get(
        "titulo"
    ):

        f.write(
            f"Proceso de admisión: "
            f"{padre['titulo']}\n\n"
        )


    # ------------------------------------------------------
    # Texto
    # ------------------------------------------------------

    if texto:

        f.write(
            texto
            +
            "\n\n"
        )


    # ------------------------------------------------------
    # Enlaces de la sección
    # ------------------------------------------------------

    escribir_enlaces(
        f,
        enlaces
    )


    # ------------------------------------------------------
    # Subsecciones
    # ------------------------------------------------------

    for subseccion in seccion.get(
        "subsecciones",
        []
    ):

        escribir_subseccion(
            f,
            subseccion,
            nivel=2
        )


    # ------------------------------------------------------
    # Bloques directos
    # ------------------------------------------------------

    for bloque in seccion.get(
        "bloques",
        []
    ):

        escribir_bloque(
            f,
            bloque,
            nivel=2
        )


# ==========================================================
# 10. CARGAR DATOS NORMALIZADOS
# ==========================================================

datos = datos_normalizados


contador = 0


# ==========================================================
# 11. GENERACIÓN
# ==========================================================

for padre in datos.get(
    "padres",
    []
):


    titulo_padre = padre.get(
        "titulo",
        ""
    )


    nombre_padre = limpiar_nombre(
        titulo_padre
    )


    if not nombre_padre:

        nombre_padre = "padre"


    # ------------------------------------------------------
    # Carpeta del padre
    # ------------------------------------------------------

    carpeta_padre = os.path.join(
        directorio_base,
        nombre_padre
    )


    os.makedirs(
        carpeta_padre,
        exist_ok=True
    )


    # ======================================================
    # DOCUMENTO PADRE
    # ======================================================

    archivo_padre = os.path.join(

        directorio_base,

        f"{nombre_padre}.md"

    )


    with open(
        archivo_padre,
        "w",
        encoding="utf-8"
    ) as f:


        escribir_metadatos(
            f,
            "padre"
        )


        # --------------------------------------------------
        # Título
        # --------------------------------------------------

        if titulo_padre:

            f.write(
                f"# {titulo_padre}\n\n"
            )


        # --------------------------------------------------
        # Introducción
        # --------------------------------------------------

        f.write(
            INTRO_DOCUMENTO
            +
            "\n\n"
        )


        # --------------------------------------------------
        # URL fuente
        # --------------------------------------------------

        url_padre = padre.get(
            "url",
            ""
        )


        if url_padre:

            f.write(
                f"Fuente oficial: "
                f"{url_padre}\n\n"
            )


        # --------------------------------------------------
        # Secciones
        # --------------------------------------------------

        nombres_padre = set()


        for i, seccion in enumerate(
            padre.get(
                "secciones",
                []
            ),
            start=1
        ):

            titulo = seccion.get(
                "titulo",
                ""
            )


            if titulo:

                f.write(
                    f"## {titulo}\n\n"
                )

            else:

                f.write(
                    f"## Sección {i}\n\n"
                )


            texto = seccion.get(
                "texto",
                ""
            )


            if texto:

                f.write(
                    texto
                    +
                    "\n\n"
                )


            escribir_enlaces(
                f,
                seccion.get(
                    "enlaces",
                    []
                )
            )


            for subseccion in seccion.get(
                "subsecciones",
                []
            ):

                escribir_subseccion(
                    f,
                    subseccion,
                    nivel=3
                )


            for bloque in seccion.get(
                "bloques",
                []
            ):

                escribir_bloque(
                    f,
                    bloque,
                    nivel=3
                )


    contador += 1


    # ======================================================
    # DOCUMENTOS POR SECCIÓN
    # ======================================================

    nombres_utilizados = set()


    for i, seccion in enumerate(
        padre.get(
            "secciones",
            []
        ),
        start=1
    ):


        nombre_seccion = generar_nombre_seccion(

            seccion.get(
                "titulo",
                ""
            ),

            i,

            nombres_utilizados

        )


        archivo_seccion = os.path.join(

            carpeta_padre,

            f"{nombre_seccion}.md"

        )


        with open(
            archivo_seccion,
            "w",
            encoding="utf-8"
        ) as f:


            escribir_metadatos(
                f,
                "seccion",
                nombre_seccion
            )


            escribir_seccion(
                f,
                seccion,
                padre
            )


        contador += 1


# ==========================================================
# 12. RESULTADO
# ==========================================================

print()
print("=" * 70)
print(
    "BLOQUE 7 - MARKDOWN BASE PARA RAG"
)
print("=" * 70)

print()
print(
    "Markdown base generado correctamente."
)

print()
print(
    "Archivos creados:",
    contador
)

print()
print(
    "Ruta:",
    directorio_base
)


BLOQUE 7 - MARKDOWN BASE PARA RAG

Markdown base generado correctamente.

Archivos creados: 9

Ruta: /content/drive/MyDrive/TFG Teleco/ADMISION/Internacional


In [37]:
# ==========================================================
# BLOQUE 8. Descarga, filtrado y extracción de recursos
#
# Admisión Internacional
# ==========================================================


import requests

from bs4 import BeautifulSoup

from urllib.parse import urlparse, urlunparse


# ==========================================================
# 1. CONFIGURACIÓN
# ==========================================================


paginas_extraidas = []


# ----------------------------------------------------------
# Dominios UPV permitidos
# ----------------------------------------------------------

DOMINIOS_UPV = {
    "www.upv.es",
    "upv.es"
}


# ----------------------------------------------------------
# URLs con tratamiento manual
# ----------------------------------------------------------

# Estas URLs se fuerzan manualmente independientemente
# del resultado del filtro de relevancia.

URLS_DESCARTAR_MANUALMENTE = {

    "http://www.upv.es/es",
    "http://www.upv.es/index-es.html"

}


URLS_ACEPTAR_MANUALMENTE = {

    "http://www.upv.es/rankings/index.html"

}


# ==========================================================
# 2. FUNCIONES AUXILIARES
# ==========================================================


def normalizar_url_final(url):

    if not url:

        return None


    try:

        p = urlparse(url)

    except Exception:

        return None


    return urlunparse(
        (
            p.scheme.lower(),
            p.netloc.lower(),
            p.path.rstrip("/") or "/",
            "",
            p.query,
            ""
        )
    )


# ----------------------------------------------------------
# Comprobar si la URL pertenece al ecosistema UPV
# ----------------------------------------------------------


def es_url_internacional(url):

    if not url:

        return False


    try:

        p = urlparse(url)

        dominio = (
            p.netloc
            .lower()
            .split(":")[0]
        )

    except Exception:

        return False


    if dominio in DOMINIOS_UPV:

        return True


    if dominio.endswith(".upv.es"):

        return True


    return False


# ----------------------------------------------------------
# Comprobar tratamiento manual de una URL
# ----------------------------------------------------------


def tratamiento_manual_url(url):

    """
    Devuelve:

        "aceptar"   -> aceptar directamente
        "descartar" -> descartar directamente
        None        -> aplicar filtro normal
    """

    url_normalizada = normalizar_url_final(url)


    if not url_normalizada:

        return None


    # ------------------------------------------------------
    # URLs que deben descartarse siempre
    # ------------------------------------------------------

    if url_normalizada in URLS_DESCARTAR_MANUALMENTE:

        return "descartar"


    # ------------------------------------------------------
    # URLs que deben aceptarse siempre
    # ------------------------------------------------------

    if url_normalizada in URLS_ACEPTAR_MANUALMENTE:

        return "aceptar"


    return None


# ----------------------------------------------------------
# Eliminar elementos que no forman parte del contenido
# ----------------------------------------------------------


def eliminar_basura(soup):

    for elemento in soup.find_all(
        [
            "script",
            "style",
            "noscript",
            "header",
            "footer",
            "nav",
            "aside",
            "form",
            "iframe"
        ]
    ):

        elemento.decompose()


# ----------------------------------------------------------
# Encontrar contenido principal
# ----------------------------------------------------------


def encontrar_contenido_principal(soup):

    selectores = [

        "main",

        "article",

        "#content",

        ".content",

        ".container",

        ".main-content"

    ]


    for selector in selectores:

        elemento = soup.select_one(
            selector
        )


        if elemento is not None:

            texto = elemento.get_text(
                " ",
                strip=True
            )


            if len(texto) >= 100:

                return elemento


    # ------------------------------------------------------
    # Último recurso: body
    # ------------------------------------------------------

    body = soup.find(
        "body"
    )


    if body is not None:

        texto = body.get_text(
            " ",
            strip=True
        )


        if len(texto) >= 100:

            return body


    return None


# ----------------------------------------------------------
# Normalización de texto
# ----------------------------------------------------------


def normalizar_texto_para_comparacion(texto):

    if not texto:

        return ""


    return " ".join(
        texto.lower().split()
    )


# ----------------------------------------------------------
# Limpieza de texto de BeautifulSoup
# ----------------------------------------------------------


def limpiar_texto_extraido(elemento):

    if elemento is None:

        return ""


    return " ".join(
        elemento.stripped_strings
    )


# ==========================================================
# 3. FILTRO DE RELEVANCIA INTERNACIONAL
# ==========================================================


def contenido_relevante_internacional(
    titulo,
    texto_enlace,
    contenido,
    url
):

    """
    Determina si una página descargada es potencialmente
    relevante para la admisión internacional.

    El filtro busca señales relacionadas con:

        - estudiantes internacionales
        - estudiantes extranjeros
        - admisión
        - acceso
        - preinscripción
        - matrícula
        - requisitos
        - documentación
        - visados
        - programas académicos

    Las excepciones manuales se gestionan fuera de esta
    función mediante tratamiento_manual_url().
    """


    titulo = titulo or ""

    texto_enlace = texto_enlace or ""

    contenido = contenido or ""

    url = url or ""


    # ------------------------------------------------------
    # Texto utilizado para la clasificación
    # ------------------------------------------------------

    texto = (
        titulo
        + " "
        + texto_enlace
        + " "
        + contenido
        + " "
        + url
    ).lower()


    # ======================================================
    # 1. INDICADORES DIRECTOS
    # ======================================================

    indicadores_directos = [

        "admisión internacional",

        "admisión de estudiantes internacionales",

        "international admission",

        "international admissions",

        "international students",

        "international student",

        "estudiantes internacionales",

        "estudiantes extranjeros",

        "alumnos extranjeros",

        "alumnado extranjero",

        "información alumnos extranjeros",

        "informacion alumnos extranjeros",

        "international applicants",

        "foreign students",

        "foreign applicants"

    ]


    for indicador in indicadores_directos:

        if indicador in texto:

            return True


    # ======================================================
    # 2. SEÑALES DE ADMISIÓN
    # ======================================================

    indicadores_admision = [

        "preinscripción",

        "preinscripcion",

        "solicitud de admisión",

        "solicitud de acceso",

        "solicitar el acceso",

        "solicitud de plaza",

        "proceso de admisión",

        "proceso de acceso",

        "requisitos de acceso",

        "requisitos de admisión",

        "criterios de admisión",

        "plazo de admisión",

        "application",

        "apply",

        "admission requirements",

        "entry requirements",

        "application process"

    ]


    coincidencias_admision = sum(

        1

        for indicador in indicadores_admision

        if indicador in texto

    )


    # ======================================================
    # 3. SEÑALES DE MATRÍCULA / TRÁMITES
    # ======================================================

    indicadores_matricula = [

        "matrícula",

        "matricula",

        "matriculación",

        "matriculacion",

        "formalizar la matrícula",

        "enrollment",

        "enrolment",

        "enrollment process",

        "registration",

        "documentación necesaria",

        "documentacion necesaria",

        "documentación requerida",

        "documentacion requerida",

        "required documents"

    ]


    coincidencias_matricula = sum(

        1

        for indicador in indicadores_matricula

        if indicador in texto

    )


    # ======================================================
    # 4. SEÑALES INTERNACIONALES
    # ======================================================

    indicadores_internacional = [

        "extranjero",

        "extranjeros",

        "international",

        "foreign",

        "visa",

        "visado",

        "student visa",

        "visado de estudiante",

        "país de origen",

        "pais de origen",

        "estudiante internacional",

        "estudiantes internacionales"

    ]


    coincidencias_internacional = sum(

        1

        for indicador in indicadores_internacional

        if indicador in texto

    )


    # ======================================================
    # 5. SEÑALES ACADÉMICAS
    # ======================================================

    indicadores_academicos = [

        "grado",

        "grados",

        "máster",

        "master",

        "máster universitario",

        "master universitario",

        "bachelor",

        "master's degree",

        "degree",

        "programa de estudios",

        "programa académico",

        "estudios universitarios"

    ]


    coincidencias_academicas = sum(

        1

        for indicador in indicadores_academicos

        if indicador in texto

    )


    # ======================================================
    # 6. REGLAS DE ACEPTACIÓN
    # ======================================================

    # ------------------------------------------------------
    # Admisión + componente internacional
    # ------------------------------------------------------

    if (

        coincidencias_admision >= 1

        and

        coincidencias_internacional >= 1

    ):

        return True


    # ------------------------------------------------------
    # Admisión + información académica
    # ------------------------------------------------------

    if (

        coincidencias_admision >= 1

        and

        coincidencias_academicas >= 1

    ):

        return True


    # ------------------------------------------------------
    # Matrícula/documentación + componente internacional
    # ------------------------------------------------------

    if (

        coincidencias_matricula >= 1

        and

        coincidencias_internacional >= 1

    ):

        return True


    # ------------------------------------------------------
    # Matrícula/documentación + información académica
    # ------------------------------------------------------

    if (

        coincidencias_matricula >= 1

        and

        coincidencias_academicas >= 1

    ):

        return True


    return False


# ----------------------------------------------------------
# Comprobar contenido mínimo
# ----------------------------------------------------------


def contenido_demasiado_corto(contenido):

    return len(
        contenido.strip()
    ) < 100


# ==========================================================
# 4. EXTRAER ENLACES DEL JSON FINAL
# ==========================================================


lista_enlaces = []


for padre in datos_normalizados.get(
    "padres",
    []
):

    titulo_padre = padre.get(
        "titulo",
        ""
    )


    # ------------------------------------------------------
    # Recorrido recursivo de bloques
    # ------------------------------------------------------

    def recorrer_bloque(
        bloque,
        seccion_origen="",
        tipo_origen=""
    ):

        # --------------------------------------------------
        # Enlaces del bloque
        # --------------------------------------------------

        for enlace in bloque.get(
            "enlaces",
            []
        ):

            url = enlace.get(
                "url",
                ""
            )


            if not url:

                continue


            lista_enlaces.append({

                "texto":
                    enlace.get(
                        "texto",
                        ""
                    ),

                "url":
                    url,

                "seccion_origen":
                    seccion_origen,

                "padre_origen":
                    titulo_padre,

                "tipo_origen":
                    tipo_origen

            })


        # --------------------------------------------------
        # Bloques internos
        # --------------------------------------------------

        for sub_bloque in bloque.get(
            "bloques",
            []
        ):

            recorrer_bloque(
                sub_bloque,
                seccion_origen,
                tipo_origen
            )


        # --------------------------------------------------
        # Subsecciones internas
        # --------------------------------------------------

        for subseccion in bloque.get(
            "subsecciones",
            []
        ):

            recorrer_bloque(
                subseccion,
                seccion_origen,
                tipo_origen
            )


    # ======================================================
    # Recorrer secciones
    # ======================================================

    for seccion in padre.get(
        "secciones",
        []
    ):

        titulo_seccion = seccion.get(
            "titulo",
            ""
        )


        tipo_seccion = seccion.get(
            "tipo",
            ""
        )


        # --------------------------------------------------
        # Enlaces directos de la sección
        # --------------------------------------------------

        for enlace in seccion.get(
            "enlaces",
            []
        ):

            url = enlace.get(
                "url",
                ""
            )


            if not url:

                continue


            lista_enlaces.append({

                "texto":
                    enlace.get(
                        "texto",
                        ""
                    ),

                "url":
                    url,

                "seccion_origen":
                    titulo_seccion,

                "padre_origen":
                    titulo_padre,

                "tipo_origen":
                    tipo_seccion

            })


        # --------------------------------------------------
        # Subsecciones
        # --------------------------------------------------

        for subseccion in seccion.get(
            "subsecciones",
            []
        ):

            recorrer_bloque(
                subseccion,
                titulo_seccion,
                tipo_seccion
            )


        # --------------------------------------------------
        # Bloques directos
        # --------------------------------------------------

        for bloque in seccion.get(
            "bloques",
            []
        ):

            recorrer_bloque(
                bloque,
                titulo_seccion,
                tipo_seccion
            )


# ==========================================================
# 5. DEDUPLICACIÓN PREVIA
# ==========================================================


urls_entrada_vistas = set()

enlaces_unicos = []


for enlace in lista_enlaces:

    url_original = enlace.get(
        "url",
        ""
    )


    if not url_original:

        continue


    url_normalizada = normalizar_url_final(
        url_original
    )


    if not url_normalizada:

        continue


    if url_normalizada in urls_entrada_vistas:

        continue


    urls_entrada_vistas.add(
        url_normalizada
    )


    enlaces_unicos.append(
        enlace
    )


print()

print(
    "Enlaces extraídos del JSON:",
    len(lista_enlaces)
)

print(
    "Enlaces únicos antes de descargar:",
    len(enlaces_unicos)
)


# ==========================================================
# 6. DEDUPLICACIÓN DE URLs FINALES
# ==========================================================


vistos_urls_finales = set()


# ==========================================================
# 7. SESIÓN HTTP
# ==========================================================


sesion = requests.Session()

sesion.headers.update(
    HEADERS
)


# ==========================================================
# 8. DESCARGA Y FILTRADO
# ==========================================================


for enlace in enlaces_unicos:

    url_original = enlace["url"]


    print()

    print("=" * 80)

    print(
        url_original
    )


    # ======================================================
    # Comprobar ámbito antes de descargar
    # ======================================================

    if not es_url_internacional(
        url_original
    ):

        print(
            "Descartado (URL fuera del ámbito UPV)"
        )

        continue


    # ======================================================
    # Descargar
    # ======================================================

    try:

        respuesta = sesion.get(

            url_original,

            timeout=20,

            allow_redirects=True

        )

        respuesta.raise_for_status()


    except Exception as e:

        print(
            "Error de descarga:",
            e
        )

        continue


    # ------------------------------------------------------
    # URL final después de redirecciones
    # ------------------------------------------------------

    url_final = normalizar_url_final(
        respuesta.url
    )


    print(
        "URL final:",
        url_final
    )


    # ------------------------------------------------------
    # Comprobar ámbito después de redirección
    # ------------------------------------------------------

    if not es_url_internacional(
        url_final
    ):

        print(
            "Descartado (URL final fuera del ámbito UPV)"
        )

        continue


    # ------------------------------------------------------
    # Evitar duplicados después de redirección
    # ------------------------------------------------------

    if url_final in vistos_urls_finales:

        print(
            "Descartado (URL final duplicada)"
        )

        continue


    vistos_urls_finales.add(
        url_final
    )


    # ======================================================
    # TRATAMIENTO MANUAL DE URL
    # ======================================================

    tratamiento = tratamiento_manual_url(
        url_final
    )


    if tratamiento == "descartar":

        print(
            "Descartado (URL excluida manualmente)"
        )

        continue


    if tratamiento == "aceptar":

        aceptar_manualmente = True

    else:

        aceptar_manualmente = False


    # ------------------------------------------------------
    # Comprobar tipo de contenido
    # ------------------------------------------------------

    content_type = respuesta.headers.get(
        "Content-Type",
        ""
    ).lower()


    if "text/html" not in content_type:

        print(
            "Descartado (recurso no HTML)"
        )

        continue


    # ======================================================
    # 9. PARSEAR HTML
    # ======================================================


    soup = BeautifulSoup(

        respuesta.text,

        "html.parser"

    )


    eliminar_basura(
        soup
    )


    # ======================================================
    # 10. CONTENIDO PRINCIPAL
    # ======================================================


    contenido_principal = (
        encontrar_contenido_principal(
            soup
        )
    )


    if contenido_principal is None:

        print(
            "Descartado "
            "(no se encontró contenido principal)"
        )

        continue


    # ======================================================
    # 11. TÍTULO
    # ======================================================


    titulo = ""


    h1 = contenido_principal.find(
        "h1"
    )


    if h1:

        titulo = limpiar_texto_extraido(
            h1
        )


    if not titulo:

        h1 = soup.find(
            "h1"
        )


        if h1:

            titulo = limpiar_texto_extraido(
                h1
            )


    if not titulo:

        titulo = enlace.get(
            "texto",
            ""
        )


    titulo = " ".join(
        titulo.split()
    )


    # ======================================================
    # 12. EXTRACCIÓN ESTRUCTURADA
    # ======================================================


    bloques = []


    for elemento in contenido_principal.find_all(

        [
            "h1",
            "h2",
            "h3",
            "h4",
            "p",
            "li",
            "table"
        ]

    ):

        texto = limpiar_texto_extraido(
            elemento
        )


        if len(texto) < 3:

            continue


        # --------------------------------------------------
        # Evitar repetir el título principal
        # --------------------------------------------------

        if (

            elemento.name == "h1"

            and normalizar_texto_para_comparacion(
                texto
            )
            ==
            normalizar_texto_para_comparacion(
                titulo
            )

        ):

            continue


        bloques.append(
            texto
        )


    # ======================================================
    # 13. ELIMINAR DUPLICADOS CONSECUTIVOS
    # ======================================================


    bloques_limpios = []


    for bloque in bloques:

        if (

            bloques_limpios

            and normalizar_texto_para_comparacion(
                bloque
            )
            ==
            normalizar_texto_para_comparacion(
                bloques_limpios[-1]
            )

        ):

            continue


        bloques_limpios.append(
            bloque
        )


    contenido = "\n\n".join(
        bloques_limpios
    )


    # ======================================================
    # 14. FILTRADO DE CONTENIDO INSUFICIENTE
    # ======================================================

    if contenido_demasiado_corto(
        contenido
    ):

        print(
            "Descartado (contenido insuficiente)"
        )

        continue


    # ======================================================
    # 15. FILTRADO DE RELEVANCIA INTERNACIONAL
    # ======================================================


    # ------------------------------------------------------
    # Si la URL no tiene tratamiento manual, se aplica
    # el filtro normal de relevancia.
    # ------------------------------------------------------

    if not aceptar_manualmente:

        if not contenido_relevante_internacional(

            titulo,

            enlace.get(
                "texto",
                ""
            ),

            contenido,

            url_final

        ):

            print(
                "Descartado "
                "(sin relevancia para admisión internacional)"
            )

            continue


    # ======================================================
    # 16. GUARDAR PÁGINA ACEPTADA
    # ======================================================


    paginas_extraidas.append({

        "url_original":
            url_original,

        "url":
            url_final,

        "titulo":
            titulo,

        "texto_enlace":
            enlace.get(
                "texto",
                ""
            ),

        "seccion_origen":
            enlace.get(
                "seccion_origen",
                ""
            ),

        "padre_origen":
            enlace.get(
                "padre_origen",
                ""
            ),

        "tipo_origen":
            enlace.get(
                "tipo_origen",
                ""
            ),

        "contenido":
            contenido

    })


    if aceptar_manualmente:

        print(
            "Página aceptada "
            "(URL aceptada manualmente)"
        )

    else:

        print(
            "Página aceptada"
        )


# ==========================================================
# 17. RESUMEN
# ==========================================================


print()

print("=" * 80)

print(
    "RESULTADO DE LA EXTRACCIÓN"
)

print("=" * 80)

print()


print(
    "Enlaces extraídos del JSON:",
    len(lista_enlaces)
)


print(
    "Enlaces únicos procesados:",
    len(enlaces_unicos)
)


print(
    "URLs finales únicas:",
    len(vistos_urls_finales)
)


print(
    "Páginas útiles:",
    len(paginas_extraidas)
)


print()


# ==========================================================
# 18. MOSTRAR PRIMERAS PÁGINAS
# ==========================================================


for pagina in paginas_extraidas[:5]:

    print(
        "=" * 80
    )


    print(
        pagina["titulo"]
    )


    print()


    print(
        "Sección:",
        pagina["seccion_origen"]
    )


    print(
        "URL:",
        pagina["url"]
    )


    print()


    print(
        pagina["contenido"][:1000]
    )


    print()


Enlaces extraídos del JSON: 13
Enlaces únicos antes de descargar: 6

http://www.upv.es/es
URL final: http://www.upv.es/index-es.html
Descartado (URL excluida manualmente)

http://www.upv.es/rankings/index.html
URL final: http://www.upv.es/rankings/index.html
Página aceptada (URL aceptada manualmente)

https://www.upv.es/pls/soalu/SIC_POLICONSULTA.BienvenidaGlobal?P_VISTA=normal&P_IDIOMA=c
URL final: https://www.upv.es/pls/soalu/SIC_POLICONSULTA.BienvenidaGlobal?P_VISTA=normal&P_IDIOMA=c
Descartado (contenido insuficiente)

https://tramitadoredu.gva.es/jwt/#/home
Descartado (URL fuera del ámbito UPV)

https://www.upv.es/entidades/SA/mastersoficiales/1183876normalc.html
URL final: https://www.upv.es/entidades/SESTU/mastersoficiales/1183876normalc.html
Página aceptada

https://www.youtube.com/watch?v=GE8F0QF_bAg
Descartado (URL fuera del ámbito UPV)

RESULTADO DE LA EXTRACCIÓN

Enlaces extraídos del JSON: 13
Enlaces únicos procesados: 6
URLs finales únicas: 4
Páginas útiles: 2

ranking d

In [38]:
# ==========================================================
# BLOQUE 9. Generación de recursos Markdown
#
# A partir de las páginas aceptadas por el BLOQUE 8
#
# Admisión Internacional
# ==========================================================


import os
import re


# ==========================================================
# 1. RUTA DE RECURSOS
# ==========================================================

directorio_recursos = os.path.join(
    ruta_programa,
    "ADMISION",
    "Internacional",
    "recursos"
)


os.makedirs(
    directorio_recursos,
    exist_ok=True
)


# ==========================================================
# 2. LIMPIEZA DE NOMBRES DE ARCHIVO
# ==========================================================

def limpiar_nombre_recurso(nombre):

    if not nombre:

        return "recurso"


    nombre = nombre.lower()


    cambios = {

        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
        "ñ": "n"

    }


    for viejo, nuevo in cambios.items():

        nombre = nombre.replace(
            viejo,
            nuevo
        )


    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )


    nombre = nombre.strip("_")


    if not nombre:

        nombre = "recurso"


    return nombre


# ==========================================================
# 3. GENERAR NOMBRE ÚNICO
# ==========================================================

def generar_nombre_recurso(
    titulo,
    indice,
    nombres_utilizados
):

    nombre_base = limpiar_nombre_recurso(
        titulo
    )


    if not nombre_base:

        nombre_base = f"recurso_{indice}"


    nombre = nombre_base


    contador = 2


    while nombre in nombres_utilizados:

        nombre = (
            f"{nombre_base}_{contador}"
        )

        contador += 1


    nombres_utilizados.add(
        nombre
    )


    return nombre


# ==========================================================
# 4. ESCRIBIR METADATOS
# ==========================================================

def escribir_metadatos_recurso(
    f,
    pagina
):

    f.write(
        "---\n"
    )


    f.write(
        "fuente: UPV\n"
    )


    f.write(
        "categoria: admision\n"
    )


    f.write(
        "nivel: internacional\n"
    )


    f.write(
        "tipo_documento: recurso\n"
    )


    if pagina.get("seccion_origen"):

        f.write(
            "seccion: "
            + str(
                pagina["seccion_origen"]
            )
            + "\n"
        )


    if pagina.get("tipo_origen"):

        f.write(
            "tipo_origen: "
            + str(
                pagina["tipo_origen"]
            )
            + "\n"
        )


    f.write(
        "---\n\n"
    )


# ==========================================================
# 5. GENERACIÓN DE RECURSOS
# ==========================================================

nombres_utilizados = set()

contador_recursos = 0


for indice, pagina in enumerate(
    paginas_extraidas,
    start=1
):


    titulo = pagina.get(
        "titulo",
        ""
    )


    contenido = pagina.get(
        "contenido",
        ""
    )


    url = pagina.get(
        "url",
        ""
    )


    texto_enlace = pagina.get(
        "texto_enlace",
        ""
    )


    seccion_origen = pagina.get(
        "seccion_origen",
        ""
    )


    padre_origen = pagina.get(
        "padre_origen",
        ""
    )


    tipo_origen = pagina.get(
        "tipo_origen",
        ""
    )


    # ------------------------------------------------------
    # Evitar recursos sin contenido
    # ------------------------------------------------------

    if not contenido.strip():

        continue


    # ------------------------------------------------------
    # Generar nombre
    # ------------------------------------------------------

    nombre_recurso = generar_nombre_recurso(

        titulo,

        indice,

        nombres_utilizados

    )


    archivo_recurso = os.path.join(

        directorio_recursos,

        f"{nombre_recurso}.md"

    )


    # ======================================================
    # CREAR MARKDOWN
    # ======================================================

    with open(

        archivo_recurso,

        "w",

        encoding="utf-8"

    ) as f:


        # --------------------------------------------------
        # METADATOS
        # --------------------------------------------------

        escribir_metadatos_recurso(

            f,

            pagina

        )


        # --------------------------------------------------
        # TÍTULO
        # --------------------------------------------------

        if titulo:

            f.write(
                f"# {titulo}\n\n"
            )

        else:

            f.write(
                "# Recurso de admisión internacional\n\n"
            )


        # --------------------------------------------------
        # FUENTE
        # --------------------------------------------------

        if url:

            f.write(
                "Fuente oficial: "
                + url
                + "\n\n"
            )


        # --------------------------------------------------
        # TEXTO DEL ENLACE
        # --------------------------------------------------

        if texto_enlace:

            f.write(
                "Enlace de origen: "
                + texto_enlace
                + "\n\n"
            )


        # --------------------------------------------------
        # PROCEDENCIA
        # --------------------------------------------------

        if padre_origen:

            f.write(
                "Página de origen: "
                + padre_origen
                + "\n\n"
            )


        if seccion_origen:

            f.write(
                "Sección de origen: "
                + seccion_origen
                + "\n\n"
            )


        if tipo_origen:

            f.write(
                "Tipo de origen: "
                + tipo_origen
                + "\n\n"
            )


        # --------------------------------------------------
        # CONTENIDO
        # --------------------------------------------------

        f.write(
            "## Contenido\n\n"
        )


        f.write(
            contenido.strip()
            +
            "\n"
        )


    contador_recursos += 1


# ==========================================================
# 6. RESULTADO
# ==========================================================

print()

print(
    "=" * 70
)

print(
    "BLOQUE 9 - RECURSOS MARKDOWN"
)

print(
    "=" * 70
)

print()

print(
    "Recursos generados:",
    contador_recursos
)

print()

print(
    "Ruta:",
    directorio_recursos
)

print()

print(
    "Los recursos corresponden exclusivamente "
    "a las páginas aceptadas por el BLOQUE 8."
)

print()


BLOQUE 9 - RECURSOS MARKDOWN

Recursos generados: 2

Ruta: /content/drive/MyDrive/TFG Teleco/ADMISION/Internacional/recursos

Los recursos corresponden exclusivamente a las páginas aceptadas por el BLOQUE 8.

